## Get filename


In [23]:
from pathlib import Path
from typing import List

def get_filenames_fs(path: str) -> List[str]:
    dir_path = Path(path) # Use '.' for the current directory
    if dir_path.is_dir():
        return [f.name for f in dir_path.iterdir() if f.is_file()]
    else:
        raise FileNotFoundError(f"Directory not found at {dir_path}")


In [24]:
file_names = get_filenames_fs(path='./data/baseline')

## Paragmatic Regex generation.
just adding pattern using or.

In [ ]:
import re

def generate_regex_for_filenames(filenames):
    """
    Generates a single regex that matches all filenames in the given list.
    Special characters in filenames are escaped.
    """
    escaped_filenames = [re.escape(filename) for filename in filenames]
    print("::", escaped_filenames)
    regex_pattern = "|".join(escaped_filenames)
    return regex_pattern

# Example usage:
filename_list = ["file1.txt", "document_2.pdf", "image (3).jpg", "data-set.csv"]
generated_regex = generate_regex_for_filenames(file_names)
print(f"Generated Regex: {generated_regex}")

## Regex generation agent using AKD
- takes in list of filenames from local fs
- generates the filename_regex, id_regex
- todo: validate the regex from the same input filenames list.
- todo: get the asset names, and then find the regex for the assets too.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from akd._base import InputSchema, OutputSchema
from akd.agents import LiteLLMInstructorBaseAgent, BaseAgentConfig
import asyncio
import nest_asyncio

nest_asyncio.apply()

from stac_pydantic import Collection, Item, ItemCollection
from stac_pydantic.links import Relations
from stac_pydantic.shared import BBox, MimeTypes

class RegexGeneratorInputSchema(InputSchema):
  """
  Input schema for the Regex Generator Agent.
  """
  filenames: list[str] = Field(
    ...,
    description="The list of filenames from where the regex is to be generated."
  )
  
class FilenameRegex(BaseModel):
  regex: str = Field(..., description="""
    A regex which it pre-filters the necessary files on a set of files.
    Our assumption is that all files in the filenames are necessary.
  """)
  reason: str = Field(..., description="""
    Reasons behind the regex creation. Provide an example to prove the reason too.
  """)
  
class IDRegex(BaseModel):
  regex: str = Field(..., description="""
    A regex which from the list of filenames separates/clusters files based on the datetime frequency.
  """)
  reason: str = Field(..., description="""
    Reasons behind the regex creation. Provide an example to prove the reason too.
  """)

class RegexGeneratorOutputSchema(OutputSchema):
  """
  Output schema for the Regex Generator agent after proper metadata are extracted
  from the input content.
  """
  filename_regex: FilenameRegex = Field(...)
  id_regex: IDRegex = Field(...)
  
class RegexGeneratorAgent(
  LiteLLMInstructorBaseAgent[
    RegexGeneratorInputSchema,
    RegexGeneratorOutputSchema
  ]
):
  """
  Agent that extracts the metadata as mentioned in the output schema from the input content about dataset.
  """
  input_schema = RegexGeneratorInputSchema
  output_schema = RegexGeneratorOutputSchema
  
async def main():
  config = BaseAgentConfig(
    model_name="ollama/llama3:8b",
    base_url="http://localhost:11434",
  )
  
  agent = RegexGeneratorAgent(
    config=config
  )
  
  file_names = get_filenames_fs(path='./data/baseline')
  inputSchema = RegexGeneratorInputSchema(filenames=file_names)
  output = await agent.arun(inputSchema)
  return output

agent_result = asyncio.run(main())
print("result is:: ", agent_result)


11:40:38 - LiteLLM:INFO: utils.py:3373 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
2025-10-16 11:40:38,213 - INFO - 
LiteLLM completion() model= gpt-4o-mini; provider = openai


result is::  filename_regex=FilenameRegex(regex='^GCHP_\\d{8}_\\d{4}z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\\.1x0\\.1_lev0\\.nc4$', reason="This regex matches filenames that start with 'GCHP_', followed by an 8-digit date (YYYYMMDD), a 4-digit time (HHMM), and ends with a specific format. For example, 'GCHP_20200719_0400z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0.1x0.1_lev0.nc4'.") id_regex=IDRegex(regex='^GCHP_\\d{8}_\\d{4}z', reason="This regex captures the unique identifier for each file, which consists of 'GCHP_' followed by an 8-digit date and a 4-digit time. For example, 'GCHP_20200719_0400z' identifies the file corresponding to July 19, 2020, at 04:00.")


In [51]:
agent_result.model_dump_json()

'{"filename_regex":{"regex":"^GCHP_\\\\d{8}_\\\\d{4}z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\\\\.1x0\\\\.1_lev0\\\\.nc4$","reason":"This regex matches filenames that start with \'GCHP_\', followed by an 8-digit date (YYYYMMDD), a 4-digit time (HHMM), and ends with a specific format. For example, \'GCHP_20200719_0400z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0.1x0.1_lev0.nc4\'."},"id_regex":{"regex":"^GCHP_\\\\d{8}_\\\\d{4}z","reason":"This regex captures the unique identifier for each file, which consists of \'GCHP_\' followed by an 8-digit date and a 4-digit time. For example, \'GCHP_20200719_0400z\' identifies the file corresponding to July 19, 2020, at 04:00."}}'

## Test

In [35]:
from typing import List

def is_regex_valid(regex: str, test_filenames: List[str]):
    """
    Tests if the provided regex is covers the provided list of (test) filenames
    """
    valid = True
    for filename in test_filenames:
        # if re.fullmatch(llm_generated_regex, filename):
        if re.fullmatch(regex, filename):
            # print(f"✅'{filename}' matches the regex.")
            pass
        else:
            # print(f"❌'{filename}' does NOT match the regex.")
            valid = False
    return valid

In [ ]:
def regex_test():
  filename_regex='GCHP_\\d{8}_\\d{4}z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\\.1x0\\.1_lev0\\.nc4'
  id_regex='GCHP_(\\d{8})_(\\d{4})z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\\.1x0\\.1_lev0\\.nc4'
  regex_list = [filename_regex, id_regex]
  file_names = get_filenames_fs(path='./data/baseline')
  for regex in regex_list:
    result = is_regex_valid(regex, file_names)
    if (not result):
      print(f"🚨🚨Test Failed for regex: {regex}")
    else:
      print(f"✅'Regex: {regex}' passed.")


In [37]:
regex_test()

✅'Regex: GCHP_\d{8}_\d{4}z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\.1x0\.1_lev0\.nc4' passed.
✅'Regex: GCHP_(\d{8})_(\d{4})z_SpeciesConcVV_O3_C180_SF4_to_CONUS_0\.1x0\.1_lev0\.nc4' passed.
